In [173]:
import numpy as np
from numba import njit

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import scipy.sparse as sp
from scipy.spatial.distance import directed_hausdorff
import scipy.linalg as la

from sklearn.linear_model import RidgeCV, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, root_mean_squared_error

import optuna
import optuna.visualization as vis
from optuna.importance import PedAnovaImportanceEvaluator

import warnings

import math

In [174]:
steps = 20000

tau_steps = 1

transient_steps_henon = int(steps * 0.1)
transient_steps_reservoir = int(steps * 0.1)

total_steps = steps + transient_steps_henon + transient_steps_reservoir + tau_steps
total_steps_after_henon = steps + transient_steps_reservoir + tau_steps

test_size = 0.2
test_steps = int(steps * test_size)

t = np.arange(0, total_steps)

# Init

In [175]:
henon_dataset = np.zeros((total_steps, 2))

rng = np.random.default_rng(42)
henon_dataset[0] = rng.random(2)

a = 1.4
b = 0.3

In [176]:
@njit
def henon_numba(steps, a=1.4, b=0.3, x0=0.0, y0=0.0):
    X = np.zeros(steps)
    Y = np.zeros(steps)
    X[0] = x0
    Y[0] = y0

    for i in range(1, steps):
        X[i] = 1 - a * X[i - 1] ** 2 + Y[i - 1]
        Y[i] = b * X[i - 1]

    return X, Y

In [177]:
henon_data_x, henon_data_y = henon_numba(total_steps)

henon_dataset = np.column_stack((henon_data_x, henon_data_y))
henon_dataset = henon_dataset[transient_steps_henon:]

In [178]:
henon_scaler = StandardScaler()
henon_scaled = henon_scaler.fit_transform(henon_dataset)

In [179]:
def henon_plot(data_list, names=None):
    fig = go.Figure()
    colors = ["white", "magenta"]

    for i, data in enumerate(data_list):
        fig.add_trace(
            go.Scattergl(
                x=data[:, 0],
                y=data[:, 1],
                mode="markers",
                name=names[i] if names else f"Dataset {i+1}",
                marker=dict(color=colors[i % len(colors)], size=1),
            )
        )

    fig.update_layout(
        plot_bgcolor="black",
        paper_bgcolor="black",
        font=dict(color="white"),
        xaxis=dict(
            showgrid=False,
            zeroline=False,
            linecolor="white",
            ticks="outside",
            tickcolor="white",
        ),
        yaxis=dict(
            showgrid=False,
            zeroline=False,
            linecolor="white",
            ticks="outside",
            tickcolor="white",
        ),
    )

    return fig

In [180]:
def r_2_plots_grid(actual_list, predicted_list, titles):
    fig = make_subplots(rows=1, cols=actual_list.shape[1], subplot_titles=titles)

    for i in range(actual_list.shape[1]):
        actual = actual_list[:, i]
        predicted = predicted_list[:, i]
        r_2 = r2_score(actual, predicted)
        col = i + 1

        fig.add_trace(go.Scatter(
            x=actual, y=predicted, mode="markers",
            name="Data", marker=dict(color="rgba(50, 50, 200, 0.5)", size=5)
        ), row=1, col=col)

        min_val, max_val = min(actual.min(), predicted.min()), max(actual.max(), predicted.max())
        fig.add_trace(go.Scatter(
            x=[min_val, max_val], y=[min_val, max_val], mode="lines", 
            name="Ideal", line=dict(color="firebrick", dash="dash")
        ), row=1, col=col)

        fig.update_xaxes(title_text="Actual", row=1, col=col)
        fig.update_yaxes(title_text=f"Predicted (R²: {r_2:.4f})", row=1, col=col)

    fig.update_layout(showlegend=False, height=500, width=1000)
    return fig

In [181]:
def generate_henon_grid(all_experiments, cols=3, plot_height=400):
    total_plots = len(all_experiments)
    rows = math.ceil(total_plots / cols)
    colors = ["white", "magenta"]

    # 1. Initialize the master subplot matrix
    fig = make_subplots(
        rows=rows,
        cols=cols,
        subplot_titles=[f"System #{i+1}" for i in range(total_plots)],
        horizontal_spacing=0.04,
        vertical_spacing=0.03,
    )

    for idx, data_list in enumerate(all_experiments):
        current_row = (idx // cols) + 1
        current_col = (idx % cols) + 1

        for i, data in enumerate(data_list):
            fig.add_trace(
                go.Scattergl(
                    x=data[:, 0],
                    y=data[:, 1],
                    mode="markers",
                    marker=dict(color=colors[i % len(colors)], size=1),
                    showlegend=False,
                ),
                row=current_row,
                col=current_col,
            )

    fig.update_layout(
        height=plot_height * rows,
        plot_bgcolor="black",
        paper_bgcolor="black",
        font=dict(color="white", size=10),
        margin=dict(t=80, b=40, l=40, r=40),
    )

    fig.update_xaxes(
        showgrid=False,
        zeroline=False,
        linecolor="white",
        ticks="outside",
        tickcolor="white",
    )
    fig.update_yaxes(
        showgrid=False,
        zeroline=False,
        linecolor="white",
        ticks="outside",
        tickcolor="white",
    )

    return fig

# Bayesian Init

In [182]:
@njit(fastmath=True, cache=True)
def compute_states(steps, henon_inputs, res_size, W_in, W_res, bias, alpha, noise):
    X_data = np.zeros((steps, res_size))
    X_data[0] = 0.0

    state = np.zeros(res_size)
    input_projections = (henon_inputs + noise) @ W_in.T

    for i in range(1, steps):
        state = (1.0 - alpha) * state + alpha * np.tanh(
            input_projections[i - 1] + W_res @ state + bias
        )
        X_data[i] = state
    return X_data

In [183]:
@njit(fastmath=True, cache=True)
def compute_closed_states(
    steps_start,
    steps_end,
    Y_pred_scaled,
    X_pred,
    W_in,
    W_res,
    bias,
    alpha,
    W_out,
    W_bias,
    res_size,
    scaler_mean,
    scaler_std,
):
    for i in range(steps_start, steps_end):
        u = Y_pred_scaled[i - 2]
        prev_state = X_pred[i - 1, :res_size]
        new_state = np.tanh(W_in @ u + W_res @ prev_state + bias)
        X_pred[i, :res_size] = (1.0 - alpha) * prev_state + alpha * new_state
        X_scaled = (X_pred[i, :res_size] - scaler_mean) / scaler_std
        Y_pred_scaled[i] = W_out @ X_scaled + W_bias
    return Y_pred_scaled

In [184]:
@njit(fastmath=True, cache=True)
def lyapunov_loss(Y_test, Y_pred, a, b, total_steps_after_henon):
    Q_test = np.ascontiguousarray(np.eye(2))
    lyapunov_sums_test = np.zeros(2)
    for i in range(len(Y_test)):
        J = np.array([[-2.0 * a * Y_test[i, 0], 1.0], [b, 0.0]])
        Z = np.ascontiguousarray(J @ Q_test)
        Q_test_raw, R = np.linalg.qr(Z)
        Q_test = np.ascontiguousarray(Q_test_raw)
        lyapunov_sums_test += np.log(np.abs(np.diag(R)))
    lyapunov_exponent_test = lyapunov_sums_test / total_steps_after_henon

    Q_pred = np.ascontiguousarray(np.eye(2))
    lyapunov_sums_pred = np.zeros(2)
    for i in range(len(Y_pred)):
        J = np.array([[-2.0 * a * Y_pred[i, 0], 1.0], [b, 0.0]])
        Z = np.ascontiguousarray(J @ Q_pred)
        Q_pred_raw, R = np.linalg.qr(Z)
        Q_pred = np.ascontiguousarray(Q_pred_raw)
        lyapunov_sums_pred += np.log(np.abs(np.diag(R)))
    lyapunov_exponent_pred = lyapunov_sums_pred / total_steps_after_henon

    le_loss = (lyapunov_exponent_test - lyapunov_exponent_pred) ** 2
    return le_loss[0]

In [185]:
def henon_closed(
    in_size,
    out_size,
    res_size,
    sparsity,
    spec_rad,
    alpha,
    input_scaling,
    bias_scaling,
    ridge_alpha,
    noise_val,
    tau_steps,
):
    rng = np.random.default_rng(42)
    bias = rng.uniform(-bias_scaling, bias_scaling, res_size)
    W_in = rng.uniform(-input_scaling, input_scaling, (res_size, in_size))
    W_res = rng.uniform(-1.0, 1.0, (res_size, res_size))
    mask = rng.random((res_size, res_size)) < sparsity
    W_res *= mask
    try:
        eigenvalues = la.eigvals(W_res)
    except Exception as e:
        print("Eigenval Issue")
        raise optuna.exceptions.TrialPruned()
    largest_eigenvalue = np.round(np.max(np.abs(eigenvalues)), decimals=12)
    W_res = W_res * (spec_rad / largest_eigenvalue)

    rng = np.random.default_rng(42)
    noise = rng.normal(
        0, noise_val, size=(len(henon_scaled) - test_steps - tau_steps, out_size)
    )
    X = compute_states(
        steps + transient_steps_reservoir - test_steps,
        henon_scaled[: -test_steps - tau_steps],
        res_size,
        W_in,
        W_res,
        bias,
        alpha,
        noise,
    )
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X[transient_steps_reservoir:])
    Y_train = henon_scaled[transient_steps_reservoir + tau_steps : -test_steps]
    scaler_mean = scaler.mean_
    scaler_std = scaler.scale_

    with warnings.catch_warnings():
        warnings.filterwarnings("error", category=Warning)
        try:
            model = Ridge(alpha=ridge_alpha, solver="auto")
            model.fit(X_train, Y_train)
        except (Warning, ValueError):
            try:
                model = Ridge(alpha=ridge_alpha, solver="svd")
                model.fit(X_train, Y_train)
            except Exception as e:
                print("Ridge nor SVD worked")
                raise optuna.exceptions.TrialPruned()

    W_out = model.coef_
    W_bias = model.intercept_

    X_pred = np.zeros((test_steps, res_size))
    Y_pred_scaled = np.zeros((test_steps, out_size))
    Y_test = henon_dataset[-test_steps:]

    u = Y_train[-2]
    prev_state = X_train[-1]
    new_state = np.tanh(W_in @ u + W_res @ prev_state + bias)
    X_pred[0] = (1 - alpha) * prev_state + alpha * new_state
    X_pred_scaled = scaler.transform(X_pred[0].reshape(1, -1))
    Y_pred_scaled[0] = model.predict(X_pred_scaled)

    u = Y_train[-1]
    prev_state = X_pred[0]
    new_state = np.tanh(W_in @ u + W_res @ prev_state + bias)
    X_pred[1] = (1 - alpha) * prev_state + alpha * new_state
    X_pred_scaled = scaler.transform(X_pred[1].reshape(1, -1))
    Y_pred_scaled[1] = model.predict(X_pred_scaled)

    compute_closed_states(
        0,
        test_steps,
        Y_pred_scaled,
        X_pred,
        W_in,
        W_res,
        bias,
        alpha,
        W_out,
        W_bias,
        res_size,
        scaler_mean,
        scaler_std,
    )

    Y_pred = henon_scaler.inverse_transform(Y_pred_scaled)

    return Y_test, Y_pred

In [186]:
def normalize_2d(data):
    min_vals = data.min(axis=0)
    max_vals = data.max(axis=0)
    return (data - min_vals) / (max_vals - min_vals)

In [187]:
def standardize_2d(data):
    mean_vals = data.mean(axis=0)
    std_vals = data.std(axis=0)
    return (data - mean_vals) / (std_vals + 1e-8)

# Lupanuv vs NRSME vs Haussdorf for res of 100

In [188]:
res_size = 100

In [189]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=res_size,
        sparsity=trial.suggest_float("sparsity", 0.01, 0.5),
        spec_rad=trial.suggest_float("spec_rad", 0.2, 2.0),
        alpha=trial.suggest_float("alpha", 0.1, 0.8),
        input_scaling=trial.suggest_float("input_scaling", 1e-2, 2.0),
        bias_scaling=trial.suggest_float("bias_scaling", 1e-2, 2.0),
        ridge_alpha=trial.suggest_float("ridge_alpha", 1e-13, 1e-3, log=True),
        noise_val=trial.suggest_float("noise_val", 1e-6, 0.1, log=True),
        tau_steps=1,
    )

    drift_penalty_x = np.sum(np.abs(Y_pred[:, 0]) > 1.5) * 5
    drift_penalty_y = np.sum(np.abs(Y_pred[:, 1]) > 0.6) * 5

    Y_test_norm = normalize_2d(Y_test)
    Y_pred_norm = normalize_2d(Y_pred)

    h_uv = directed_hausdorff(Y_test_norm, Y_pred_norm)[0]
    h_vu = directed_hausdorff(Y_pred_norm, Y_test_norm)[0]

    rmse = root_mean_squared_error(Y_test, Y_pred)
    std = np.std(Y_test)
    nrmse = rmse / std

    return (
        max(h_uv, h_vu) + drift_penalty_x + drift_penalty_y,
        nrmse,
        lyapunov_loss(Y_test, Y_pred, a, b, total_steps_after_henon),
    )


study = optuna.create_study(directions=["minimize", "minimize", "minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=1000, n_jobs=-1)

[Optuna] Processing Trial #49...

KeyboardInterrupt: 

In [ ]:
data = []
for i, trial in enumerate(study.best_trials):
    print(f"\rProcessing Trial {i}/{len(study.best_trials)}...", end="", flush=True)
    params = trial.params
    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=res_size,
        sparsity=params["sparsity"],
        spec_rad=params["spec_rad"],
        alpha=params["alpha"],
        input_scaling=params["input_scaling"],
        bias_scaling=params["bias_scaling"],
        ridge_alpha=params["ridge_alpha"],
        noise_val=params["noise_val"],
        tau_steps=1,
    )
    data.append((Y_test, Y_pred))

Processing Trial 45/46...

In [ ]:
generate_henon_grid(data, cols=5).show()

# Lupanuv vs NRSME vs Haussdorf for res of 150

In [ ]:
res_size = 150

In [ ]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=res_size,
        sparsity=trial.suggest_float("sparsity", 0.01, 0.5),
        spec_rad=trial.suggest_float("spec_rad", 0.2, 2.0),
        alpha=trial.suggest_float("alpha", 0.1, 0.8),
        input_scaling=trial.suggest_float("input_scaling", 1e-2, 2.0),
        bias_scaling=trial.suggest_float("bias_scaling", 1e-2, 2.0),
        ridge_alpha=trial.suggest_float("ridge_alpha", 1e-13, 1e-3, log=True),
        noise_val=trial.suggest_float("noise_val", 1e-6, 0.1, log=True),
        tau_steps=1,
    )

    drift_penalty_x = np.sum(np.abs(Y_pred[:, 0]) > 1.5) * 5
    drift_penalty_y = np.sum(np.abs(Y_pred[:, 1]) > 0.6) * 5

    Y_test_norm = normalize_2d(Y_test)
    Y_pred_norm = normalize_2d(Y_pred)

    h_uv = directed_hausdorff(Y_test_norm, Y_pred_norm)[0]
    h_vu = directed_hausdorff(Y_pred_norm, Y_test_norm)[0]

    rmse = root_mean_squared_error(Y_test, Y_pred)
    std = np.std(Y_test)
    nrmse = rmse / std

    return (
        max(h_uv, h_vu) + drift_penalty_x + drift_penalty_y,
        nrmse,
        lyapunov_loss(Y_test, Y_pred, a, b, total_steps_after_henon),
    )


study = optuna.create_study(directions=["minimize", "minimize", "minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=1000, n_jobs=-1)

[Optuna] Processing Trial #999...

In [ ]:
data = []
for i, trial in enumerate(study.best_trials):
    print(f"\r Processing Trial {i}/{len(study.best_trials)}...", end="", flush=True)
    params = trial.params
    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=res_size,
        sparsity=params["sparsity"],
        spec_rad=params["spec_rad"],
        alpha=params["alpha"],
        input_scaling=params["input_scaling"],
        bias_scaling=params["bias_scaling"],
        ridge_alpha=params["ridge_alpha"],
        noise_val=params["noise_val"],
        tau_steps=1,
    )
    data.append((Y_test, Y_pred))

 Processing Trial 53/54...

In [ ]:
generate_henon_grid(data, cols=5).show()

# Lupanuv vs NRSME vs Haussdorf for res of 200

In [ ]:
res_size = 200

In [ ]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=res_size,
        sparsity=trial.suggest_float("sparsity", 0.01, 0.5),
        spec_rad=trial.suggest_float("spec_rad", 0.2, 2.0),
        alpha=trial.suggest_float("alpha", 0.1, 0.8),
        input_scaling=trial.suggest_float("input_scaling", 1e-2, 2.0),
        bias_scaling=trial.suggest_float("bias_scaling", 1e-2, 2.0),
        ridge_alpha=trial.suggest_float("ridge_alpha", 1e-13, 1e-3, log=True),
        noise_val=trial.suggest_float("noise_val", 1e-6, 0.1, log=True),
        tau_steps=1,
    )

    drift_penalty_x = np.sum(np.abs(Y_pred[:, 0]) > 1.5) * 5
    drift_penalty_y = np.sum(np.abs(Y_pred[:, 1]) > 0.6) * 5

    Y_test_norm = normalize_2d(Y_test)
    Y_pred_norm = normalize_2d(Y_pred)
    h_uv = directed_hausdorff(Y_test_norm, Y_pred_norm)[0]
    h_vu = directed_hausdorff(Y_pred_norm, Y_test_norm)[0]

    rmse = root_mean_squared_error(Y_test, Y_pred)
    std = np.std(Y_test)
    nrmse = rmse / std

    return (
        max(h_uv, h_vu) + drift_penalty_x + drift_penalty_y,
        nrmse,
        lyapunov_loss(Y_test, Y_pred, a, b, total_steps_after_henon),
    )


study = optuna.create_study(directions=["minimize", "minimize", "minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=1000, n_jobs=-1)

[Optuna] Processing Trial #999...

In [ ]:
data = []
for i, trial in enumerate(study.best_trials):
    print(f"\rProcessing Trial {i}/{len(study.best_trials)}...", end="", flush=True)
    params = trial.params
    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=res_size,
        sparsity=params["sparsity"],
        spec_rad=params["spec_rad"],
        alpha=params["alpha"],
        input_scaling=params["input_scaling"],
        bias_scaling=params["bias_scaling"],
        ridge_alpha=params["ridge_alpha"],
        noise_val=params["noise_val"],
        tau_steps=1,
    )
    data.append((Y_test, Y_pred))

Processing Trial 87/88...

In [ ]:
generate_henon_grid(data, cols=5).show()

# Lupanuv vs NRSME vs Haussdorf for res of 250

In [ ]:
res_size = 250

In [ ]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=res_size,
        sparsity=trial.suggest_float("sparsity", 0.01, 0.5),
        spec_rad=trial.suggest_float("spec_rad", 0.2, 2.0),
        alpha=trial.suggest_float("alpha", 0.1, 0.8),
        input_scaling=trial.suggest_float("input_scaling", 1e-2, 2.0),
        bias_scaling=trial.suggest_float("bias_scaling", 1e-2, 2.0),
        ridge_alpha=trial.suggest_float("ridge_alpha", 1e-13, 1e-3, log=True),
        noise_val=trial.suggest_float("noise_val", 1e-6, 0.1, log=True),
        tau_steps=1,
    )

    drift_penalty_x = np.sum(np.abs(Y_pred[:, 0]) > 1.5) * 5
    drift_penalty_y = np.sum(np.abs(Y_pred[:, 1]) > 0.6) * 5

    Y_test_norm = normalize_2d(Y_test)
    Y_pred_norm = normalize_2d(Y_pred)

    h_uv = directed_hausdorff(Y_test_norm, Y_pred_norm)[0]
    h_vu = directed_hausdorff(Y_pred_norm, Y_test_norm)[0]

    rmse = root_mean_squared_error(Y_test, Y_pred)
    std = np.std(Y_test)
    nrmse = rmse / std

    return (
        max(h_uv, h_vu) + drift_penalty_x + drift_penalty_y,
        nrmse,
        lyapunov_loss(Y_test, Y_pred, a, b, total_steps_after_henon),
    )


study = optuna.create_study(directions=["minimize", "minimize", "minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=1000, n_jobs=-1)

[Optuna] Processing Trial #999...

In [ ]:
data = []
for i, trial in enumerate(study.best_trials):
    print(f"\rProcessing Trial {i}/{len(study.best_trials)}...", end="", flush=True)
    params = trial.params
    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=res_size,
        sparsity=params["sparsity"],
        spec_rad=params["spec_rad"],
        alpha=params["alpha"],
        input_scaling=params["input_scaling"],
        bias_scaling=params["bias_scaling"],
        ridge_alpha=params["ridge_alpha"],
        noise_val=params["noise_val"],
        tau_steps=1,
    )
    data.append((Y_test, Y_pred))

Processing Trial 55/56...

In [ ]:
generate_henon_grid(data, cols=5).show()

# Lupanuv vs NRSME vs Haussdorf for res of 250 first couple of NRMSE

In [ ]:
res_size = 250

In [ ]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=res_size,
        sparsity=trial.suggest_float("sparsity", 0.01, 0.5),
        spec_rad=trial.suggest_float("spec_rad", 0.2, 2.0),
        alpha=trial.suggest_float("alpha", 0.1, 0.8),
        input_scaling=trial.suggest_float("input_scaling", 1e-2, 2.0),
        bias_scaling=trial.suggest_float("bias_scaling", 1e-2, 2.0),
        ridge_alpha=trial.suggest_float("ridge_alpha", 1e-13, 1e-3, log=True),
        noise_val=trial.suggest_float("noise_val", 1e-6, 0.1, log=True),
        tau_steps=1,
    )

    drift_penalty_x = np.sum(np.abs(Y_pred[:, 0]) > 1.5) * 5
    drift_penalty_y = np.sum(np.abs(Y_pred[:, 1]) > 0.6) * 5

    Y_test_norm = normalize_2d(Y_test)
    Y_pred_norm = normalize_2d(Y_pred)

    h_uv = directed_hausdorff(Y_test_norm, Y_pred_norm)[0]
    h_vu = directed_hausdorff(Y_pred_norm, Y_test_norm)[0]

    rmse = root_mean_squared_error(Y_test[:75], Y_pred[:75])
    std = np.std(Y_test[:75])
    nrmse = rmse / std

    return (
        max(h_uv, h_vu) + drift_penalty_x + drift_penalty_y,
        nrmse,
        lyapunov_loss(Y_test, Y_pred, a, b, total_steps_after_henon),
    )


study = optuna.create_study(directions=["minimize", "minimize", "minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=1000, n_jobs=-1)

[Optuna] Processing Trial #999...

In [ ]:
data = []
for i, trial in enumerate(study.best_trials):
    print(f"\r Processing Trial {i}/{len(study.best_trials)}...", end="", flush=True)
    params = trial.params
    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=res_size,
        sparsity=params["sparsity"],
        spec_rad=params["spec_rad"],
        alpha=params["alpha"],
        input_scaling=params["input_scaling"],
        bias_scaling=params["bias_scaling"],
        ridge_alpha=params["ridge_alpha"],
        noise_val=params["noise_val"],
        tau_steps=1,
    )
    data.append((Y_test, Y_pred))

 Processing Trial 55/56...

In [ ]:
generate_henon_grid(data, cols=5).show()

In [ ]:
henon_plot(data[-1]).show()

# Lupanuv vs NRSME for res of 250

In [ ]:
res_size = 250

In [ ]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=res_size,
        sparsity=trial.suggest_float("sparsity", 0.01, 0.5),
        spec_rad=trial.suggest_float("spec_rad", 0.2, 2.0),
        alpha=trial.suggest_float("alpha", 0.1, 0.8),
        input_scaling=trial.suggest_float("input_scaling", 1e-2, 2.0),
        bias_scaling=trial.suggest_float("bias_scaling", 1e-2, 2.0),
        ridge_alpha=trial.suggest_float("ridge_alpha", 1e-13, 1e-3, log=True),
        noise_val=trial.suggest_float("noise_val", 1e-6, 0.1, log=True),
        tau_steps=1,
    )

    rmse = root_mean_squared_error(Y_test[:15], Y_pred[:15])
    std = np.std(Y_test[:15])
    nrmse = rmse / std

    return (
        nrmse,
        lyapunov_loss(Y_test, Y_pred, a, b, total_steps_after_henon),
    )


study = optuna.create_study(directions=["minimize", "minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=1000, n_jobs=-1)

[Optuna] Processing Trial #999...

In [ ]:
data = []
for i, trial in enumerate(study.best_trials):
    print(f"\r Processing Trial {i}/{len(study.best_trials)}...", end="", flush=True)
    params = trial.params
    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=res_size,
        sparsity=params["sparsity"],
        spec_rad=params["spec_rad"],
        alpha=params["alpha"],
        input_scaling=params["input_scaling"],
        bias_scaling=params["bias_scaling"],
        ridge_alpha=params["ridge_alpha"],
        noise_val=params["noise_val"],
        tau_steps=1,
    )
    data.append((Y_test, Y_pred))

 Processing Trial 6/7...

In [ ]:
generate_henon_grid(data, cols=5).show()

# Hauffdwon standardized vs NRSME for res of 250

In [ ]:
res_size = 250

In [ ]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=res_size,
        sparsity=trial.suggest_float("sparsity", 0.01, 0.5),
        spec_rad=trial.suggest_float("spec_rad", 0.2, 2.0),
        alpha=trial.suggest_float("alpha", 0.1, 0.8),
        input_scaling=trial.suggest_float("input_scaling", 1e-2, 2.0),
        bias_scaling=trial.suggest_float("bias_scaling", 1e-2, 2.0),
        ridge_alpha=trial.suggest_float("ridge_alpha", 1e-13, 1e-3, log=True),
        noise_val=trial.suggest_float("noise_val", 1e-6, 0.1, log=True),
        tau_steps=1,
    )

    drift_penalty_x = np.sum(np.abs(Y_pred[:, 0]) > 1.5) * 5
    drift_penalty_y = np.sum(np.abs(Y_pred[:, 1]) > 0.6) * 5

    Y_test_norm = standardize_2d(Y_test)
    Y_pred_norm = standardize_2d(Y_pred)

    h_uv = directed_hausdorff(Y_test_norm, Y_pred_norm)[0]
    h_vu = directed_hausdorff(Y_pred_norm, Y_test_norm)[0]

    rmse = root_mean_squared_error(Y_test[:15], Y_pred[:15])
    std = np.std(Y_test[:15])
    nrmse = rmse / std

    return (
        max(h_uv, h_vu) + drift_penalty_x + drift_penalty_y,
        nrmse,
    )


study = optuna.create_study(directions=["minimize", "minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=1000, n_jobs=-1)

[Optuna] Processing Trial #999...

In [ ]:
data = []
for i, trial in enumerate(study.best_trials):
    print(f"\r Processing Trial {i}/{len(study.best_trials)}...", end="", flush=True)
    params = trial.params
    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=res_size,
        sparsity=params["sparsity"],
        spec_rad=params["spec_rad"],
        alpha=params["alpha"],
        input_scaling=params["input_scaling"],
        bias_scaling=params["bias_scaling"],
        ridge_alpha=params["ridge_alpha"],
        noise_val=params["noise_val"],
        tau_steps=1,
    )
    data.append((Y_test, Y_pred))

 Processing Trial 14/15...

In [ ]:
generate_henon_grid(data, cols=5).show()

# Hauffdwon vs NRSME for res of 250

In [ ]:
res_size = 250

In [ ]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=res_size,
        sparsity=trial.suggest_float("sparsity", 0.01, 0.5),
        spec_rad=trial.suggest_float("spec_rad", 0.2, 2.0),
        alpha=trial.suggest_float("alpha", 0.1, 0.8),
        input_scaling=trial.suggest_float("input_scaling", 1e-2, 2.0),
        bias_scaling=trial.suggest_float("bias_scaling", 1e-2, 2.0),
        ridge_alpha=trial.suggest_float("ridge_alpha", 1e-13, 1e-3, log=True),
        noise_val=trial.suggest_float("noise_val", 1e-6, 0.1, log=True),
        tau_steps=1,
    )

    drift_penalty_x = np.sum(np.abs(Y_pred[:, 0]) > 1.5) * 5
    drift_penalty_y = np.sum(np.abs(Y_pred[:, 1]) > 0.6) * 5

    Y_test_norm = normalize_2d(Y_test)
    Y_pred_norm = normalize_2d(Y_pred)

    h_uv = directed_hausdorff(Y_test_norm, Y_pred_norm)[0]
    h_vu = directed_hausdorff(Y_pred_norm, Y_test_norm)[0]

    rmse = root_mean_squared_error(Y_test[:15], Y_pred[:15])
    std = np.std(Y_test[:15])
    nrmse = rmse / std

    return (
        max(h_uv, h_vu) + drift_penalty_x + drift_penalty_y,
        nrmse,
    )


study = optuna.create_study(directions=["minimize", "minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=1000, n_jobs=-1)

[Optuna] Processing Trial #999...

In [ ]:
data = []
for i, trial in enumerate(study.best_trials):
    print(f"\r Processing Trial {i}/{len(study.best_trials)}...", end="", flush=True)
    params = trial.params
    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=res_size,
        sparsity=params["sparsity"],
        spec_rad=params["spec_rad"],
        alpha=params["alpha"],
        input_scaling=params["input_scaling"],
        bias_scaling=params["bias_scaling"],
        ridge_alpha=params["ridge_alpha"],
        noise_val=params["noise_val"],
        tau_steps=1,
    )
    data.append((Y_test, Y_pred))

 Processing Trial 14/15...

In [ ]:
generate_henon_grid(data, cols=5).show()

# Lupanuv vs NRSME vs Haussdorf for res of 250 first couple of NRMSE

In [ ]:
res_size = 250

In [ ]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=res_size,
        sparsity=trial.suggest_float("sparsity", 0.01, 0.5),
        spec_rad=trial.suggest_float("spec_rad", 0.2, 2.0),
        alpha=trial.suggest_float("alpha", 0.1, 0.8),
        input_scaling=trial.suggest_float("input_scaling", 1e-2, 2.0),
        bias_scaling=trial.suggest_float("bias_scaling", 1e-2, 2.0),
        ridge_alpha=trial.suggest_float("ridge_alpha", 1e-13, 1e-3, log=True),
        noise_val=trial.suggest_float("noise_val", 1e-6, 0.1, log=True),
        tau_steps=1,
    )

    drift_penalty_x = np.sum(np.abs(Y_pred[:, 0]) > 1.5) * 5
    drift_penalty_y = np.sum(np.abs(Y_pred[:, 1]) > 0.6) * 5

    Y_test_norm = standardize_2d(Y_test)
    Y_pred_norm = standardize_2d(Y_pred)

    h_uv = directed_hausdorff(Y_test_norm, Y_pred_norm)[0]
    h_vu = directed_hausdorff(Y_pred_norm, Y_test_norm)[0]

    rmse = root_mean_squared_error(Y_test[:70], Y_pred[:70])
    std = np.std(Y_test[:70])
    nrmse = rmse / std

    return (
        max(h_uv, h_vu) + drift_penalty_x + drift_penalty_y,
        nrmse,
        lyapunov_loss(Y_test, Y_pred, a, b, total_steps_after_henon),
    )


study = optuna.create_study(directions=["minimize", "minimize", "minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=1000, n_jobs=-1)

[Optuna] Processing Trial #999...

In [ ]:
data = []
for i, trial in enumerate(study.best_trials):
    print(f"\r Processing Trial {i}/{len(study.best_trials)}...", end="", flush=True)
    params = trial.params
    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=res_size,
        sparsity=params["sparsity"],
        spec_rad=params["spec_rad"],
        alpha=params["alpha"],
        input_scaling=params["input_scaling"],
        bias_scaling=params["bias_scaling"],
        ridge_alpha=params["ridge_alpha"],
        noise_val=params["noise_val"],
        tau_steps=1,
    )
    data.append((Y_test, Y_pred))

 Processing Trial 85/86...

In [ ]:
generate_henon_grid(data, cols=5).show()